# Phase 10 — Explainable AI (SHAP)
**Problem Statement ID 26184:** Predictive Analytics Framework for Cybercrime Complaints to Forecast Likely Cash Withdrawal Locations

## Objective
Provide transparent Explainable AI (XAI) for the primary XGBoost predictive model:
1. Global feature importance (Mean Absolute SHAP values)
2. Local feature attribution for individual predictions across risk categories
3. Directional feature contributions (positive vs negative impact on predicted withdrawal risk)
4. Comparison between XGBoost Gain and SHAP Attribution
5. Strict mathematical consistency verification


In [ ]:
import json
import warnings
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from IPython.display import Image, display

warnings.filterwarnings('ignore')

BASE_DIR = Path('..').resolve()
DATA_DIR = BASE_DIR / 'data' / 'processed'
OUTPUTS_DIR = BASE_DIR / 'outputs'
FIG_DIR = OUTPUTS_DIR / 'figures'
MODELS_DIR = BASE_DIR / 'models'
PRED_DIR = OUTPUTS_DIR / 'predictions'


## 1. Load Model & Pipeline
Load the frozen Phase 7 XGBoost pipeline and inspect its components.

In [ ]:
pipeline = joblib.load(MODELS_DIR / 'xgboost_cybercrime_model.pkl')
preprocessor = pipeline.named_steps['preprocessor']
classifier = pipeline.named_steps['classifier']

print(f"Pipeline: {pipeline}")
print(f"Classifier: {type(classifier)}")


## 2. Load Model Metadata & Input Profile
Inspect feature counts and training configuration.

In [ ]:
with open(MODELS_DIR / 'phase7_xgboost_metadata.json', 'r') as f:
    metadata = json.load(f)

input_profile = pd.read_csv(OUTPUTS_DIR / 'phase10_model_input_profile.csv')
display(input_profile)


## 3. Load Explanation Data
Load validation and test datasets and inspect sample sizes.

In [ ]:
val_df = pd.read_csv(DATA_DIR / 'validation.csv')
test_df = pd.read_csv(DATA_DIR / 'test.csv')
preds_df = pd.read_csv(PRED_DIR / 'xgboost_test_predictions.csv')

sampling_report = pd.read_csv(OUTPUTS_DIR / 'phase10_shap_sampling_report.csv')
display(sampling_report)


## 4. Prepare Transformed Feature Matrix & Feature Names
Transform raw features through the frozen `preprocessor` without refitting.

In [ ]:
feature_cols = [c for c in val_df.columns if c not in ['case_id', 'future_withdrawal']]
X_val = val_df[feature_cols]
X_val_trans = preprocessor.transform(X_val)

mapping_df = pd.read_csv(OUTPUTS_DIR / 'phase10_feature_name_mapping.csv')
clean_feature_names = mapping_df['transformed_feature'].tolist()

print(f"Transformed matrix shape: {X_val_trans.shape}")
print(f"Total mapped features: {len(clean_feature_names)}")
display(mapping_df.head(10))


## 5. Create SHAP TreeExplainer
Initialize `shap.TreeExplainer` directly on the XGBoost classifier.

In [ ]:
explainer = shap.TreeExplainer(classifier)
base_value = explainer.expected_value
if isinstance(base_value, np.ndarray):
    base_value = float(base_value[0])

print(f"Explainer: {type(explainer)}")
print(f"Expected base value (log-odds margin): {base_value:.6f}")
print(f"Base probability (sigmoid of base value): {1.0 / (1.0 + np.exp(-base_value)):.4f}")


## 6. Calculate SHAP Values
Compute additive Shapley contributions for the validation dataset.

In [ ]:
shap_values_val = explainer(X_val_trans)
shap_values_val.feature_names = clean_feature_names
print(f"Calculated SHAP values shape: {shap_values_val.values.shape}")


## 7. Global Feature Importance
Calculate mean absolute SHAP values across all validation records.

In [ ]:
global_shap_df = pd.read_csv(OUTPUTS_DIR / 'phase10_global_shap_importance.csv')
top20_df = pd.read_csv(OUTPUTS_DIR / 'phase10_top_features.csv')

print("Top 15 Features by Global SHAP Importance:")
display(top20_df.head(15))


## 8. SHAP Summary Bar Chart
Visualize global mean absolute feature impacts.

In [ ]:
display(Image(filename=str(FIG_DIR / 'phase10_shap_summary_bar.png')))


## 9. SHAP Beeswarm Plot
Visualize feature value distributions versus positive/negative contribution direction.

In [ ]:
display(Image(filename=str(FIG_DIR / 'phase10_shap_beeswarm.png')))


## 10. Local Explanations for Representative Records
Inspect top positive and negative feature contributions for specific cases across risk tiers.

In [ ]:
local_explanations_df = pd.read_csv(OUTPUTS_DIR / 'phase10_local_explanations.csv')
print(f"Total local explanations: {len(local_explanations_df)}")
display(local_explanations_df.head(20))


## 11. Local SHAP Waterfall Plots
Waterfall plots illustrating how feature attributions sum from the base value to the model margin.

In [ ]:
print("LOW Risk Local Explanation Waterfall:")
display(Image(filename=str(FIG_DIR / 'shap_local_low_risk.png')))

print("MODERATE Risk Local Explanation Waterfall:")
display(Image(filename=str(FIG_DIR / 'shap_local_moderate_risk.png')))

print("HIGH Risk Local Explanation Waterfall:")
display(Image(filename=str(FIG_DIR / 'shap_local_high_risk.png')))


## 12. XGBoost Gain vs. SHAP Importance Comparison
Analyze the divergence between tree-split Gain purity and overall distribution-level SHAP impact.

In [ ]:
comp_df = pd.read_csv(OUTPUTS_DIR / 'phase10_feature_importance_comparison.csv')
display(comp_df.head(20))


## 13. Mathematical Consistency & Quality Validation
Confirm that:
$$\text{base\_value} + \sum \text{SHAP}_i = \text{model output margin}$$

In [ ]:
cons_df = pd.read_csv(OUTPUTS_DIR / 'phase10_shap_consistency_report.csv')
print("Sample Consistency Checks:")
display(cons_df.head(10))

qual_df = pd.read_csv(OUTPUTS_DIR / 'phase10_explanation_validation.csv')
print("Quality Audit:")
display(qual_df)


## 14. Leakage & Sensitive Data Audits
Verify zero leakage and no exposure of sensitive credentials or PII.

In [ ]:
leak_audit = pd.read_csv(OUTPUTS_DIR / 'phase10_leakage_audit.csv')
display(leak_audit)

sens_audit = pd.read_csv(OUTPUTS_DIR / 'phase10_sensitive_data_audit.csv')
display(sens_audit)


## 15. Conclusions & Next Steps
1. **Model Transparency:** SHAP provides granular, mathematically faithful explanations for both population-level behavior and individual complaints.
2. **Key Attribution Drivers:** Historical category activity (`previous_activity_by_crime_category`), location-level rates (`previous_activity_by_location`), rolling window bursts, and financial magnitude dominate risk assignments.
3. **Operational Clarity:** High-risk predictions can be clearly articulated to human analysts with positive and negative contributing factors.
4. **Governance Integrity:** Zero leakage and zero PII exposure confirmed.

**Phase 10 is COMPLETE. Ready for Phase 11 — Spatial Hotspot & Geographic Cluster Analysis.**
